# The Machine Learning Part of the ImprovingCMEs

In [ ]:
# the imports
import polars as pl
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn import linear_model
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from datetime import timedelta, datetime, date
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
import sys

# SETTINGS
################################################################################
pl.Config.set_tbl_cols(-1)      # polars settings to show all the columns
pl.Config.set_tbl_rows(-1)      # polars settings to show all the rows

which_craft = "AB" # this can be A, B, or AB

address = "../../../Data/ImprovingCMEs2/"
what_order = "linear"

outfile = f"ML_LR_results_{which_craft}_{what_order}.txt"

minutes = 20                    # how many last minutes to use
################################################################################
# HELPER FUNCTIONS
################################################################################
# to read times from files
def read_time_from_file(filename):
    with open(filename, 'r') as f:
        last_line = f.readlines()[-1]
    last_line = last_line.split()
    time = last_line[0]
    #make it datetime object
    time = datetime.strptime(time, '%Y/%m/%dT%H:%M')
    return time

################################################################################
# the main code 

# make df with the columns needed 
df_out = pl.DataFrame(schema = {"CME_num": pl.String, 
                                "Actual_TT": pl.Float64, 
                                "Seed_TT": pl.Float64, 
                                "ML_TT": pl.Float64, 
                                "Seed_error": pl.Float64, 
                                "ML_error": pl.Float64})

# open the .txt file to write the ML results 
with open(outfile, 'w') as f:
    f.write('CME_num,Actual_TT,Seed_TT,ML_TT,Seed_error,ML_error\n')

cmes = ['01_2010-04-03', '02_2010-05-23', '03_2010-08-01', '04_2011-09-06', 
        '05_2011-09-13', '06_2011-10-22', '07_2012-01-19', '08_2012-03-07', 
        '09_2012-06-14', '10_2012-07-03', '11_2012-07-12' , '12_2012-09-27', 
        '13_2012-10-05']

# read in the observed linear fit
obs_linear_A_fit = pl.read_csv(address + 'training_data/obs_coefficients_' + what_order + '_A_v2.txt')
obs_linear_B_fit = pl.read_csv(address + 'training_data/obs_coefficients_' + what_order + '_B_v2.txt')

# the dictionaries to store the metrics
maes = {}
mes = {}
stds = {}

for cme in cmes:
    # get the cme eruption time
    erupt_file = address + "erupt_and_arrival_times/" + cme + "/Erupt_time.txt"
    erupt_time = read_time_from_file(erupt_file)

    # get the cme arrival time 
    arrival_file = address + "erupt_and_arrival_times/" + cme + "/Arrival_time.txt"
    arrival_time = read_time_from_file(arrival_file)

    # what is the cme travel time in hours
    cme_obs_travel_time = (arrival_time - erupt_time).total_seconds() / 3600

    # read the data from the train file
    data_A = pl.read_csv(address + 'training_data/Train_diff_' + cme + '_A_v2.txt')
    data_B = pl.read_csv(address + 'training_data/Train_diff_' + cme + '_B_v2.txt')

    # convert string to datetime object
    data_A = data_A.with_columns(
        pl.col("Time").str.to_datetime(format="%Y-%m-%d %H:%M:%S")
    )

    data_B = data_B.with_columns(
        pl.col("Time").str.to_datetime(format="%Y-%m-%d %H:%M:%S")
    )

    # also grab the seed travel time here. look for the travel_time columnm of ensemble member 11
    seed_travel_time = data_A.filter(pl.col("ensemble_member") == 11).select(pl.col("Travel_time")).item(0, 0)
    
    # grab the last minutes of data for both A and B
    data_A = data_A.filter(
        pl.col("Time") >= (pl.col("Time").max() - pl.duration(minutes = minutes))
    ).sort("Time")

    data_B = data_B.filter(
        pl.col("Time") >= (pl.col("Time").max() - pl.duration(minutes = minutes))
    ).sort("Time")

    # unique times for both A and B
    unique_times_A = data_A["Time"].unique().to_list()
    unique_times_B = data_B["Time"].unique().to_list()

    for i in range(minutes):
        # pick the data frame A at the smallest
        data_filtered_A = data_A.filter(
            pl.col("Time") == unique_times_A[i]
        )
        data_filtered_B = data_B.filter(
            pl.col("Time") == unique_times_B[i]
        )

        print(data_filtered_A)
        print(data_filtered_B)

    # separate features and targets
    X = pl.concat(
        [data_A[["EA_diff_A", "EA_diff_B"]]],
        how = "horizontal"
    )

    X = X[["EA_diff_A", "EA_diff_B"]]
    
    # target
    y = data_diff[["Travel_time"]]

    # change to numpy arrays before proceeding
    y = y.to_numpy().ravel()

    # make the ML model
    # model = linear_model.LassoLarsCV(cv = 2)
    # model = linear_model.RidgeCV(alphas = [0.1, 1.0, 10.0])
    # model = linear_model.ElasticNet(alpha=0.01, l1_ratio=0.5) 
    # model = linear_model.Lasso(alpha=0.01)
    # model = linear_model.BayesianRidge()
    # model = linear_model.RANSACRegressor()
    model = linear_model.TheilSenRegressor()

    # train the model
    model.fit(X, y)

    # now to inference - pass them in as ['EA_diff_A', 'EA_diff_B', 'b_A', 'b_B', 'm_A', 'm_B']
    X_inference = pl.DataFrame([[0.0], [0.0]],
            schema = ['EA_diff_A', 'EA_diff_B'])

    prediction = model.predict(X_inference)[0] # to have it as a float because model.predict() returns an array

    # calculate the errors for analysis later
    seed_error = cme_obs_travel_time - seed_travel_time
    ML_error = cme_obs_travel_time - prediction

    # make the new row and concatenate with df_out
    new_inference = pl.DataFrame([{"CME_num": cme, "Actual_TT": cme_obs_travel_time, "Seed_TT": seed_travel_time, "ML_TT": prediction, "Seed_error": seed_error, "ML_error": ML_error}])

    df_out = pl.concat([df_out, new_inference])

    # now write in the file 
    with open(outfile, 'a') as f:
        f.write(cme+','+str(round(cme_obs_travel_time,2))+','+str(round(seed_travel_time,2))+','+str(round(prediction,2))+','+str(round(seed_error,2))+','+str(round(ML_error,2))+'\n')

# the for loop is finished here, now to work on the overall metrics of me, mae, and std
me_df = df_out.select(
    pl.col(["Seed_error", "ML_error"]).mean()
)
mae_df = df_out.select(
    pl.col(["Seed_error", "ML_error"]).abs().mean()
)
std_dev = df_out.select(
    pl.col(["Seed_error", "ML_error"]).std()
)

with open(outfile, 'a') as f:
    f.write("Mean Absolute Seed Error: "+str(round(mae_df["Seed_error"][0],2))+'\n')
    f.write("Mean Absolute ML Error: "+str(round(mae_df["ML_error"][0],2))+'\n')
    f.write("Mean Seed Error: "+str(round(me_df["Seed_error"][0],2))+'\n')
    f.write("Mean ML error: "+str(round(me_df["ML_error"][0],2))+'\n')
    f.write("Standard Deviation of Seed Error: "+str(round(std_dev["Seed_error"][0],2))+', Standard Deviation of ML Error: '+str(round(std_dev["ML_error"][0],2))+'\n')

shape: (21, 5)
┌─────────────────────┬─────────────────┬─────────────────────┬────────────┬─────────────┐
│ Time                ┆ ensemble_member ┆ Time_since_eruption ┆ EA_diff_A  ┆ Travel_time │
│ ---                 ┆ ---             ┆ ---                 ┆ ---        ┆ ---         │
│ datetime[μs]        ┆ i64             ┆ f64                 ┆ f64        ┆ f64         │
╞═════════════════════╪═════════════════╪═════════════════════╪════════════╪═════════════╡
│ 2010-04-04 22:06:00 ┆ 1               ┆ 36.85               ┆ -10.378437 ┆ 60.75       │
│ 2010-04-04 22:06:00 ┆ 2               ┆ 36.85               ┆ -9.852308  ┆ 60.75       │
│ 2010-04-04 22:06:00 ┆ 3               ┆ 36.85               ┆ -9.352308  ┆ 59.75       │
│ 2010-04-04 22:06:00 ┆ 4               ┆ 36.85               ┆ -8.826179  ┆ 58.75       │
│ 2010-04-04 22:06:00 ┆ 5               ┆ 36.85               ┆ -8.34005   ┆ 58.75       │
│ 2010-04-04 22:06:00 ┆ 6               ┆ 36.85               ┆ -7.85005   

SystemExit: 

/Users/syedraza/cmeuncerpy/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
